In [35]:
import pandas as pd
import numbers as np
import category_encoders as ce
from sklearn.preprocessing import StandardScaler, LabelEncoder


df = pd.read_csv('../dataset/car_prices.csv', on_bad_lines='skip')

In [36]:
import math
import os
from decimal import Decimal, ROUND_HALF_DOWN
brand_mapping = {
            'landrover': 'land rover', 'land rover': 'land rover',
            'mercedes': 'mercedes-benz', 'mercedes-b': 'mercedes-benz',
            'vw': 'volkswagen', 'volkswagen': 'volkswagen',
            'gmc truck': 'gmc', 'gmc': 'gmc',
            'ford truck': 'ford', 'ford tk': 'ford', 'ford': 'ford',
            'dodge tk': 'dodge', 'dodge': 'dodge',
            'chev truck': 'chevrolet', 'chevrolet': 'chevrolet',
            'hyundai tk': 'hyundai', 'hyundai': 'hyundai',
            'mazda tk': 'mazda', 'mazda': 'mazda'
        }
output_dir = '../dataset'
os.makedirs(output_dir, exist_ok=True)

In [37]:
def categorize_trim(trim):
    if trim is None or (isinstance(trim, float) and math.isnan(trim)):
        return 'other'

    t = str(trim).lower().strip()

    categories = {
        'special edition': [
            'se', 'limited', 'lt', 'le', 'sel', 'ltz', '1lt', 'lt fleet', 'sle', '2lt', 'es', 'lt1',
            '2.5i premium pzev', 'unlimited sahara', '3,5 se', 'sle-1', 'king ranch', 'wolfsburg edition pzev',
            'se v6', 'special edition', 'zx4 se', 'sle-2', 'sle1', '2ss', 'ultimate', 'dx', 'el limited',
            'le v6', 'signature', 'se fleet', 'tdi', 'heat', 'lt3', '2.5i limited', 'sle 1500', '1500 sle',
            'wolfsburg edition', 'passion coupe', 'se1', 'sel plus'
        ],
        'touring': ['touring', 'slt', 'i touring', 'slt-1', 'slt-2', 'touring-l', 'grand touring', 'custom',
                    'lt 1500', 'slt 1500', 'i grand touring', 's grand touring', 'gtp'],
        'luxury': ['lx', 'xlt', 'gls', 'ex', 'ex-l', 'lariat', 'titanium', 'luxury', 'premium', 'xle', '+',
                   'denali', 'cxl', 'xe', '2.0 t premium quattro', 'laramie', 'platinum', 'technology package',
                   'premier', 'deluxe', 'cargo vn xlt', 'lx-p', 'gxe', 'gls 1.8t', 'glk350', 'gl450', 'gle',
                   'lx-s'],
        'sport': ['ls', 's', '2,5 s', 'sport', 'sv', '3,5 sv', 'sl', 'ls fleet', 'gt', '3,5 s', 'i sport', 'xls',
                  'sr5', '1,8 s', 'st', '3.2', '1500 ls', 'gs', '2,0 sr', '3,5 sl', '2.0t', 'sx',
                  'c300 sport 4matic', '2,0 s', 'c300 sport', 'stx', '1,6 s plus', '3.0si', 'sr', 'gt premium',
                  's pzev', 'v8', '1,8 sl', 'performance', 'supercharged', 's sport', 'turbo', 'prerunner v6',
                  '1,6 s', 'quattro', 'ls 3500', '3.5 sr', '2,0 sl', '3.2 quattro', 'sl500', 'gts', 'c350 sport',
                  'xle v6', 'sl 550', 'ex v6', 'sport pzev', 'r350', 'sl2', 'sle v6'],
        'base': ['base', 'sxt', 'xl', 'laredo', 'se pzev', 'r/t', 'hybrid', 'ce', 'american value', 'sxt fleet',
                 'cx', 'ltz fleet', 'tdi', 'crew', 'ltz 1500', 'rt', 'express', 'mainstreet', 'overland', 'eco',
                 'standard', 'fx35', 'fx4', 'fx2', 'comfort', 'value leader', 'convenience', 'easy',
                 'value package', 'journey']
    }

    for category, keywords in categories.items():
        if any(kw in t for kw in keywords):
            return category

    return 'other'

In [38]:
def categorize_body(carrozzeria):
    categories = {
        "sedan": ["sedan", "g sedan"],
        "suv": ["suv"],
        "hatchback": ["hatchback"],
        "van": ["minivan", "van", "e-series van", "transit van", "promaster cargo van", "ram van"],
        "coupé": ["coupe", "g coupe", "genesis coupe", "koup", "cts coupe", "elantra coupe", "q60 coupe",
                  "g37 coupe", "cts-v coupe"],
        "cabriolet": ["convertible", "g convertible", "beetle convertible", "q60 convertible",
                         "g37 convertible"],
        "station wagon": ["wagon", "tsx sport wagon", "cts wagon", "cts-v wagon"],
        "pickup": ["regular cab", "regular-cab", "extended cab", "king cab", "access cab", "xtracab", "cab plus",
                   "cab plus 4", "crew cab", "supercab", "quad cab", "double cab", "crewmax cab", "mega cab", "supercrew"],
    }

    for category, values in categories.items():
        if carrozzeria in values:
            return category

    return "other"

In [39]:
def preprocess(data, output_filename="processed_data.csv"):
    data = data.drop(columns=['vin', 'seller', 'saledate', 'state', 'mmr'], errors='ignore')
    data = data.rename(columns={
        "year": "anno produzione",
        "make": "marca",
        "model": "modello",
        "trim": "allestimento",
        "body": "carrozzeria",
        "transmission": "trasmissione",
        "condition": "condizione",
        "odometer": "chilometraggio",
        "color": "colorazione",
        "interior": "colore interni",
        "sellingprice": "prezzo"
    })

    soglia = 500
    counts = data['marca'].value_counts()
    valori_validi = counts[counts >= soglia].index
    data = data[data['marca'].isin(valori_validi)]

    data = data[data['prezzo'] > 500]

    str_cols = data.select_dtypes(include=['object', 'string']).columns
    data[str_cols] = data[str_cols].apply(lambda x: x.str.lower().str.strip())

    data['marca'] = data['marca'].replace(brand_mapping)

    data['condizione'] = data['condizione'].apply(
        lambda x: int(Decimal(x).to_integral_value(rounding=ROUND_HALF_DOWN)) if pd.notnull(x) else pd.NA
    )

    data['chilometraggio'] = data['chilometraggio'].apply(
        lambda x: int(Decimal(x).to_integral_value(rounding=ROUND_HALF_DOWN)) if pd.notnull(x) else pd.NA
    )

    data.loc[data['allestimento'] == data['modello'], 'allestimento'] = 'base'

    data.loc[data['colorazione'] == '—', 'colorazione'] = None
    data.loc[data['colore interni'] == '—', 'colore interni'] = None

    data['allestimento'] = data['allestimento'].apply(categorize_trim)
    data['carrozzeria'] = data['carrozzeria'].apply(categorize_body)

    output_path = os.path.join(output_dir, output_filename)
    data.to_csv(output_path, index=False)

    return output_path

In [40]:
pathToRead = preprocess(df)
df = pd.read_csv(pathToRead)
df

,anno produzione,marca,modello,allestimento,carrozzeria,trasmissione,condizione,chilometraggio,colorazione,colore interni,prezzo
0,2015,kia,sorento,luxury,suv,automatic,5.0,16639.0,white,black,21500
1,2015,kia,sorento,luxury,suv,automatic,5.0,9393.0,white,beige,21500
2,2014,bmw,3 series,special edition,sedan,automatic,4.0,1331.0,gray,black,30000
3,2015,volvo,s60,other,sedan,automatic,4.0,14282.0,white,black,27750
4,2014,bmw,6 series gran coupe,other,sedan,automatic,4.0,2641.0,gray,black,67000
...,...,...,...,...,...,...,...,...,...,...,...
537709,2015,kia,k900,luxury,sedan,NaN,4.0,18255.0,silver,black,33000
537710,2012,ram,2500,other,pickup,automatic,5.0,54393.0,white,black,30800
537711,2012,bmw,x5,other,suv,automatic,5.0,50561.0,black,black,34000
537712,2015,nissan,altima,sport,sedan,automatic,4.0,16658.0,white,black,11100


In [41]:
from sklearn.model_selection import train_test_split
# Separazione in features e target
X = df.drop(columns=['prezzo'])
y = df['prezzo']

# Suddivisione in training e test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print('Valori mancanti train set: ' + str(X_train.isnull().sum().sum()))
print('Valori mancanti test set: ' + str(X_test.isnull().sum().sum()))

Valori mancanti train set: 80459
Valori mancanti test set: 34073


In [42]:
def data_preparation_train(X_train, y_train):
    calculate_modes_and_means(X_train)
    print("Shape before data_imputation:", X_train.shape[0])
    print("Shape of y_train:", y_train.shape[0])
    X_train = data_imputation(X_train)
    print("Shape after data_imputation:", X_train.shape[0])
    X_train = feature_engineer(X_train)
    print("Shape after feature_engineer:", X_train.shape[0])
    y_train = y_train[X_train.index]
    X_train = encode_train(X_train, y_train)
    X_train = normalize_train(X_train)
    X_train = remove_outliers(X_train)
    X_train = select_features_train(X_train)

    return X_train

In [43]:
def data_preparation_test(X_test):
    X_test = data_imputation(X_test)
    print("Shape before data_imputation:", X_test.shape[0])
    X_test = data_imputation(X_test)
    print("Shape after data_imputation:", X_test.shape[0])
    X_test = feature_engineer(X_test)
    print("Shape after feature_engineer:", X_test.shape[0])
    X_test = encode_test(X_test)
    X_test = normalize_test(X_test)
    X_test = select_features_test(X_test)

    return X_test

In [44]:
group_modes = {}
mean_condizione_by_anno = {}
mean_chilometraggio_by_anno = {}
mean_chilometraggio_by_eta = {}
mean_condizione_by_eta = {}
overall_modes = {}
overall_condizione_mean = None
overall_chilometraggio_mean = None

In [45]:
def calculate_modes_and_means(X_train):
    global group_modes, mean_condizione_by_anno, mean_chilometraggio_by_anno, mean_chilometraggio_by_eta, mean_condizione_by_eta, overall_modes, overall_condizione_mean, overall_chilometraggio_mean

    X = X_train.copy()
    group_modes['carrozzeria'] = X.groupby('modello')['carrozzeria'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()
    group_modes['trasmissione'] = X.groupby('modello')['trasmissione'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()
    group_modes['colorazione'] = X.groupby('marca')['colorazione'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()
    group_modes['colore interni'] = X.groupby('marca')['colore interni'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()
    group_modes['modello'] = X.groupby('marca')['modello'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()
    group_modes['marca'] = X.groupby('modello')['marca'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()

    # Calcola le medie per anno
    mean_condizione_by_anno = X.groupby('anno produzione')['condizione'].mean().to_dict()
    mean_chilometraggio_by_anno = X.groupby('anno produzione')['chilometraggio'].mean().to_dict()

    mean_condizione_by_eta = {
        2015 - anno: media for anno, media in mean_condizione_by_anno.items()
    }

    mean_chilometraggio_by_eta = {
        2015 - anno: media for anno, media in mean_chilometraggio_by_anno.items()
    }

    # Calcola le mode complessive e le medie dopo imputazione
    overall_modes = {
        'carrozzeria': X['carrozzeria'].mode()[0],
        'trasmissione': X['trasmissione'].mode()[0],
        'colorazione': X['colorazione'].mode()[0],
        'colore interni': X['colore interni'].mode()[0],
        'modello': X['modello'].mode()[0],
        'marca': X['marca'].mode()[0]
    }

    # Calcola le medie complessive dopo imputazione per anno
    temp_condizione = X['condizione'].fillna(X['anno produzione'].map(mean_condizione_by_anno))
    overall_condizione_mean = temp_condizione.mean()

    temp_chilometraggio = X['chilometraggio'].fillna(X['anno produzione'].map(mean_chilometraggio_by_anno))
    overall_chilometraggio_mean = temp_chilometraggio.mean()


In [46]:
def data_imputation(X_train):
    global group_modes, mean_condizione_by_anno, mean_chilometraggio_by_anno, mean_chilometraggio_by_eta, mean_condizione_by_eta, overall_modes, overall_condizione_mean, overall_chilometraggio_mean

    X = X_train.copy()

    # Drop delle righe dove sia "Marca" che "Modello" sono null
    X = X.dropna(subset=["marca", "modello"], how="all")


    if 'trasmissione' in X.columns:
        X['trasmissione'] = X['trasmissione'].fillna(X['modello'].map(group_modes['trasmissione'])).fillna(
            overall_modes['trasmissione'])

    # Applica l'imputazione basata sui gruppi
    X['marca'] = X['marca'].fillna(X['marca'].map(group_modes['marca'])).fillna(overall_modes['marca'])
    X['modello'] = X['modello'].fillna(X['modello'].map(group_modes['modello'])).fillna(overall_modes['modello'])

    X['carrozzeria'] = X['carrozzeria'].fillna(X['modello'].map(group_modes['carrozzeria'])).fillna(overall_modes['carrozzeria'])

    X['colorazione'] = X['colorazione'].fillna(X['marca'].map(group_modes['colorazione'])).fillna(overall_modes['colorazione'])
    X['colore interni'] = X['colore interni'].fillna(X['marca'].map(group_modes['colore interni'])).fillna(overall_modes['colore interni'])

    X['allestimento'] = X['allestimento'].fillna('base')

    if('anno produzione' not in X.columns):
        # Imputa condizione e chilometraggio
        X['condizione'] = X['condizione'].fillna(X['età'].map(mean_condizione_by_eta)).fillna(overall_condizione_mean)
        X['chilometraggio'] = X['chilometraggio'].fillna(X['età'].map(mean_chilometraggio_by_eta)).fillna(overall_chilometraggio_mean)
    else:
        X['condizione'] = X['condizione'].fillna(X['anno produzione'].map(mean_condizione_by_anno)).fillna(
            overall_condizione_mean)
        X['chilometraggio'] = X['chilometraggio'].fillna(X['anno produzione'].map(mean_chilometraggio_by_anno)).fillna(
            overall_chilometraggio_mean)


    return X

In [47]:
def feature_engineer(data):
    anno_rif = 2015
    data['età'] = anno_rif - data['anno produzione']
    data.drop(columns=['anno produzione'], inplace=True)

    return data

In [48]:
label_encoder = LabelEncoder()
catboost_encoder = ce.CatBoostEncoder()
marca_means = None
global_mean = None
known_makes = None
known_models = None
cat_cols = None

In [49]:
def encode_train(X_train, y_train):
    global marca_means, global_mean, known_makes, known_models, cat_cols, label_encoder, catboost_encoder
    
    if 'trasmissione' in X_train.columns:
        #Codifica 'trasmissione' con LabelEncoder
        X_train['trasmissione'] = label_encoder.fit_transform(X_train['trasmissione'])

    # Colonne categoriche da codificare con CatBoostEncoder
    cat_cols = [col for col in X_train.select_dtypes(include=['object', 'category']).columns if col != 'trasmissione']
    
    # Fit CatBoostEncoder
    encoded = catboost_encoder.fit_transform(X_train[cat_cols], y_train)
    
    # Prepara sostituzioni per modelli non visti
    marca_means = X_train.join(y_train.rename('target')).groupby('marca')['target'].mean()
    global_mean = y_train.mean()
    known_makes = X_train['marca'].unique()
    known_models = X_train['modello'].unique()

    X_train[cat_cols] = encoded
    return X_train

In [50]:
def encode_test(X_test):
    global marca_means, global_mean, known_makes, known_models, cat_cols, label_encoder, catboost_encoder

    X = X_test.copy()

    if 'trasmissione' in X.columns:
        # Applica LabelEncoder a 'trasmissione'
        X['trasmissione'] = label_encoder.transform(X['trasmissione'])

    # Applica CatBoostEncoder alle altre colonne categoriche
    encoded = catboost_encoder.transform(X[cat_cols])

    # Gestisci modelli non presenti nel training set
    new_models_mask = ~X['modello'].isin(known_models)
    if new_models_mask.any():
        replacements = X.loc[new_models_mask, 'marca'].map(marca_means).fillna(global_mean)
        encoded.loc[new_models_mask, 'modello'] = replacements.values

    X[cat_cols] = encoded
    return X

In [51]:
scaler = StandardScaler()
numeric_cols = None

In [52]:
def normalize_train(X_train):
    global scaler, numeric_cols

    numeric_cols = X_train.select_dtypes(include=['number']).columns
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

    return X_train

In [53]:
def normalize_test(X_test):
    global scaler, numeric_cols

    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    return X_test

In [54]:
def remove_outliers(X_train):
    threshold = 3
    numeric_cols = X_train.select_dtypes(include=['number']).columns
    z_scores = (X_train[numeric_cols] - X_train[numeric_cols].mean()) / X_train[numeric_cols].std()
    mask = (z_scores.abs() <= threshold).all(axis=1)
    return X_train[mask]

In [55]:
threshold_variance = 0.1
features_to_keep = None

In [56]:
def select_features_train(X_train):
    global threshold_variance, features_to_keep
    variances = X_train.var()
    features_to_keep = variances[variances > threshold_variance].index.tolist()

    return X_train[features_to_keep]

In [57]:
def select_features_test(X_test):
    global features_to_keep

    return X_test[features_to_keep]

In [58]:
X_train = data_preparation_train(X_train, y_train)
y_train = y_train.loc[X_train.index]
X_test = data_preparation_test(X_test)
y_test = y_test.loc[X_test.index]

Shape before data_imputation: 376399
Shape of y_train: 376399
Shape after data_imputation: 376399
Shape after feature_engineer: 376399
Shape before data_imputation: 161315
Shape after data_imputation: 161315
Shape after feature_engineer: 161315


In [59]:
X_train

,marca,modello,allestimento,carrozzeria,condizione,chilometraggio,colorazione,colore interni,età
370378,0.004505,0.003090,0.017349,0.006465,0.616239,-0.816747,0.009198,0.007240,-1.000455
229362,0.004505,0.003090,0.017349,0.006465,1.664665,-1.039734,0.009198,1.176892,-0.732425
387435,0.004505,0.003090,0.017349,1.058902,-0.432187,-0.612846,0.009198,2.481153,-1.000455
145008,0.004505,0.003090,0.464085,-0.440885,-1.480613,-0.937608,-2.660808,0.625988,-0.464395
106422,0.635459,0.003090,2.432325,0.184744,0.616239,-1.110970,2.867803,0.007240,-0.464395
...,...,...,...,...,...,...,...,...,...
137337,-0.613135,-0.427996,-1.192499,-0.717261,0.616239,0.067196,-0.426073,-0.118561,-0.732425
110268,-0.420794,-0.258109,-0.027712,0.978653,-1.480613,1.126184,-0.952308,-1.167990,0.607725
259178,1.621139,-1.130222,2.198157,-0.717249,0.616239,-0.208959,-0.952399,0.799879,1.411815
131932,1.620887,0.963074,2.198028,-0.717208,0.616239,-0.850848,1.155380,0.799863,-0.732425


In [60]:
X_test

,marca,modello,allestimento,carrozzeria,condizione,chilometraggio,colorazione,colore interni,età
143263,0.204491,0.018182,-0.027783,0.978565,1.664665,-0.865921,0.718261,-1.168026,-1.000455
336773,0.204491,0.399382,-0.027783,-0.717186,0.616239,-0.338668,-0.952421,0.799878,-1.000455
124520,0.204491,0.018182,-0.027783,0.978565,0.616239,-0.960805,0.718261,-1.168026,-1.000455
237169,-1.073456,-0.940705,-0.027783,-0.717186,0.616239,-0.478537,-0.426146,0.799878,-0.732425
402422,-0.420854,-0.753094,-0.027783,-1.500656,-0.432187,-0.077587,-0.426146,0.799878,-0.732425
...,...,...,...,...,...,...,...,...,...
390890,1.887813,0.600547,0.004528,-0.717186,0.616239,-0.747096,-0.952421,0.799878,-0.464395
448066,0.204491,0.801051,-0.027783,1.647391,-0.432187,0.960867,-0.426146,-1.168026,0.071665
177951,1.592611,-0.869229,-0.027783,-0.717186,1.664665,0.548720,0.090905,-1.168026,1.143785
102019,0.204491,-0.330110,-0.027783,-0.717186,-1.480613,1.302307,0.945166,3.053177,1.947875


In [61]:
X_train.isnull().sum()

marca             0
modello           0
allestimento      0
carrozzeria       0
condizione        0
chilometraggio    0
colorazione       0
colore interni    0
età               0
dtype: int64

In [62]:
X_test.isnull().sum()

marca             0
modello           0
allestimento      0
carrozzeria       0
condizione        0
chilometraggio    0
colorazione       0
colore interni    0
età               0
dtype: int64